In [8]:
import numpy as np
import pandas as pd
import zarr
import os
import matplotlib.pyplot as plt
plt.style.use('../libs/my_style.mplstyle')

import os
import sys

sys.path.append(os.path.abspath(".."))

def vec(df):
    # 1. 前処理: frameでソートしておく（groupby内での処理を減らすため）
    df = df.sort_values(['particle', 'frame'])

    # 2. 差分計算（各グループの最初の一行はNaNになる）
    # diff() は pandas のメソッドを使うとインデックスが維持されるので安全です
    dx = df.groupby('particle')['x'].diff()
    dy = df.groupby('particle')['y'].diff()

    # 3. 速度ベクトルと単位ベクトルの計算
    v = np.sqrt(dx**2 + dy**2)

    df['theta'] = dx/v + 1j * dy/v

    return df


folder = '/Volumes/My Passport/Sasaki/MTsingleBeads'

MT_path = 'MTtrack.csv'
beads_path = 'beads_tracks.csv'

pathM = os.path.join(folder, "20260122/exp001", MT_path)
pathB = os.path.join(folder, "20260122/exp001", beads_path)

dfM = pd.read_csv(pathM)
dfM = vec(dfM)

dfB = pd.read_csv(pathB)
dfB = vec(dfB)

def calculate_polar_order(df):
    """
    各フレームごとのポーラー度（Order Parameter）を算出する
    """
    # 1. 各フレームごとに複素ベクトルの平均を取る
    # thetaには既に単位ベクトル（dx/v + i*dy/v）が入っている前提
    group_avg = df.groupby('frame')['theta'].mean()
    
    # 2. 平均ベクトルの絶対値（長さ）を計算する
    # これが 1 に近いほど揃っており、0 に近いほどバラバラ
    polar_order = group_avg.abs()

    # 各フレームごとの粒子の数をカウント
    # 'theta' が NaN でないもの（速度が計算できているもの）だけを数えるのが実用的です
    counts = df.dropna(subset=['theta']).groupby('frame')['particle'].count()
    
    return polar_order, counts


In [20]:
x = dfB['x']
y = dfB['y']
frame = dfB['frame']

In [ ]:
Lum = 5
scale = 0.11
L = Lum/scale

for frame, data in dfB.groupby('frame'):
    current = dfM[dfM['frame']==frame]
    for _, i in data.groupby('particle'):
        center_x = i['x']
        center_y = i['y']
        local = current[(center_x-L/2<current['x']) & (current['x']<center_x+L/2) & (center_y-L/2<current['y']) & (current['y']<center_y+L/2)]

ValueError: Can only compare identically-labeled Series objects

In [26]:
data

,y,x,mass,size,ecc,signal,raw_mass,ep,frame,particle,theta
2563,418.757460,1738.956168,153263.141176,23.999844,0.172782,73.564706,1680724.0,NaN,500,3,-0.456903+0.889516j
2564,1077.027820,982.267745,148051.800000,19.957388,0.051602,83.576471,1581365.0,NaN,500,5,-0.974092+0.226153j
2565,1408.193077,274.503679,101225.470588,18.529909,0.250345,98.811765,1418917.0,NaN,500,6,0.132968-0.991120j
2562,1476.117861,1925.211631,135048.694118,19.530049,0.166701,78.788235,1614426.0,NaN,500,7,0.171378-0.985205j
